# insurance-credibility

**Credibility models for UK non-life insurance pricing.**

This notebook covers two distinct problems that pricing teams conflate:
group credibility (scheme pricing, large accounts) and individual
policy experience rating (NCD adjustments, Bayesian updating).

The Bühlmann-Straub formula gives the optimal weight to assign a scheme's own
loss history. Too much weight and you price noise. Too little and you leave money
on the table. The `StaticCredibilityModel` extends this to individual policies
with multi-year claims histories.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/insurance-credibility/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip install -q insurance-credibility polars numpy

## Part 1: Group credibility (scheme pricing)

We create a synthetic portfolio of 8 fleet schemes with 3 years of loss history.
Schemes vary in size (exposure) and inherent riskiness.
The Bühlmann-Straub formula gives each scheme a credibility factor Z
that determines how much weight to place on the scheme's own experience
vs the portfolio mean.

In [ ]:
import numpy as np
import polars as pl
from insurance_credibility import BuhlmannStraub

rng = np.random.default_rng(2024)

# 8 schemes × 3 years of experience
# Scheme true loss rates drawn from portfolio prior (between-variance = tau^2)
n_schemes = 8
portfolio_mean = 0.12   # 12% loss rate market rate
tau = 0.04              # between-scheme standard deviation
true_rates = rng.normal(portfolio_mean, tau, n_schemes).clip(0.02, 0.35)

rows = []
for i, rate in enumerate(true_rates):
    scheme = f"SCH{i+1:02d}"
    # Vary exposure by scheme: small schemes (50 vehicles) to large (500 vehicles)
    base_exp = rng.integers(50, 500)
    for year in [2022, 2023, 2024]:
        exp = base_exp * rng.uniform(0.9, 1.1)
        observed = rng.normal(rate, 0.03 / np.sqrt(exp / 100))
        rows.append({"scheme": scheme, "year": year,
                     "loss_rate": round(max(0, observed), 4),
                     "exposure": round(exp, 1)})

df = pl.DataFrame(rows)
print(f"{df.shape[0]} scheme-year observations across {n_schemes} schemes")
df.head(6)

In [ ]:
bs = BuhlmannStraub()
bs.fit(df, group_col="scheme", period_col="year",
       loss_col="loss_rate", weight_col="exposure")

print(f"Portfolio mean (mu):         {bs.mu_:.4f}")
print(f"Between-variance (tau^2):    {bs.tau2_:.6f}")
print(f"Within-variance / wt (sigma^2/w — Buehlmann k): {bs.k_:.2f}")
print()

# Credibility factors and blended premiums
results = pl.DataFrame({
    "scheme":           list(bs.z_.keys()),
    "credibility_Z":    [round(v, 3) for v in bs.z_.values()],
    "blended_premium":  [round(v, 4) for v in bs.premiums_.values()],
}).sort("credibility_Z", descending=True)

print("Credibility factors (Z=1 means full own experience, Z=0 means revert to market):")
print(results)

## Part 2: Individual policy experience rating

Now we model individual commercial policies with multi-year claims histories.
`ClaimsHistory` packages up the policy-level data.
`StaticCredibilityModel` fits the Bühlmann-Straub parameters at individual level
and produces a credibility factor (CF) for each policy.

CF > 1 means the policy has worse experience than the prior; CF < 1 means better.

In [ ]:
from insurance_credibility import ClaimsHistory, StaticCredibilityModel

# Construct policy histories — 3 policy types
histories = [
    # Good risk: 5 years, never claimed
    ClaimsHistory("POL001", periods=[1, 2, 3, 4, 5],
                  claim_counts=[0, 0, 0, 0, 0],
                  exposures=[1.0, 1.0, 1.0, 1.0, 1.0],
                  prior_premium=350.0),
    # Average risk: 3 years, 1 claim
    ClaimsHistory("POL002", periods=[1, 2, 3],
                  claim_counts=[0, 1, 0],
                  exposures=[1.0, 1.0, 1.0],
                  prior_premium=420.0),
    # Bad risk: 4 years, 3 claims
    ClaimsHistory("POL003", periods=[1, 2, 3, 4],
                  claim_counts=[1, 0, 2, 1],
                  exposures=[1.0, 1.0, 1.0, 1.0],
                  prior_premium=380.0),
    # New risk: 1 year, no claims yet
    ClaimsHistory("POL004", periods=[1],
                  claim_counts=[0],
                  exposures=[0.8],
                  prior_premium=400.0),
]

# Fit on the population of histories
model = StaticCredibilityModel()
model.fit(histories)

print(f"Portfolio heterogeneity (tau^2): {model.tau2_:.5f}")
print(f"Buehlmann k (signal-to-noise):  {model.k_:.2f}")
print()

# Predict credibility factors
for h in histories:
    cf = model.predict(h)
    n_claims = sum(h.claim_counts)
    total_exp = sum(h.exposures)
    print(f"  {h.policy_id}  exp={total_exp:.1f}  claims={n_claims}  CF={cf:.3f}  "
          f"adjusted_premium={h.prior_premium * cf:.0f}")

## Part 3: Dynamic experience rating

For long-history policies, the static model ignores trend. The
`DynamicPoissonGammaModel` uses a Poisson-gamma state-space model
where the policy's latent risk level can drift over time.

In [ ]:
from insurance_credibility import DynamicPoissonGammaModel

# Long history: 8 years, underlying risk increasing
long_history = ClaimsHistory(
    "POL005",
    periods=list(range(1, 9)),
    claim_counts=[0, 0, 1, 0, 1, 1, 2, 1],  # rising trend
    exposures=[1.0] * 8,
    prior_premium=400.0,
)

dyn = DynamicPoissonGammaModel()
dyn.fit([long_history] + histories)

cf_static = model.predict(long_history)
cf_dynamic = dyn.predict(long_history)

print(f"Static credibility factor:  {cf_static:.3f}")
print(f"Dynamic credibility factor: {cf_dynamic:.3f}")
print()
print("Dynamic gives higher CF because recent years (with more claims)")
print("are weighted more heavily than early years (no claims).")

## What you should see

- Large schemes get credibility factors close to 1 (trust their own experience);
  small schemes get low Z (revert to portfolio mean).
- POL001 (5 clean years) gets CF < 1 — discount vs prior premium.
  POL003 (4 years, 4 claims) gets CF > 1 — load vs prior premium.
  POL004 (new risk) gets CF ≈ 1 — no experience to deviate from prior.
- The dynamic model gives a higher CF than the static model for POL005
  because recent years are recency-weighted.

## Next steps

- **`HierarchicalBuhlmannStraub`** — nested structures: sector → district → scheme
- **`SurrogateModel`** — importance-sampled surrogate for fast prediction on large books
- **`balance_calibrate()`** — ensure the credibility adjustments are balance-neutral

**GitHub:** https://github.com/burning-cost/insurance-credibility  
**PyPI:** https://pypi.org/project/insurance-credibility/